# Simple Linear Regression: Marketing ROI Analysis

## Project Goal
Analyze a marketing dataset to identify which marketing channel (TV, Radio, or Social Media) has the strongest correlation with Sales and provide ROI-based recommendations for budget allocation.

## Project Tasks
1. Load and explore the dataset; handle missing values
2. Perform exploratory data analysis (EDA) with visualizations
3. Identify the independent variable most correlated with Sales
4. Build an OLS regression model using statsmodels
5. Create diagnostic plots to test Linearity, Normality, and Homoscedasticity
6. Interpret R-squared, coefficients, and p-values in business context
7. Formulate a clear ROI-based recommendation for marketing budget allocation

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.graphics.gofplots import ProbPlot
import warnings
warnings.filterwarnings('ignore')

# Set style for better-looking plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Load and Explore the Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('marketing_and_sales_data_evaluate_lr.csv')

# Display first few rows
print("First 5 rows of the dataset:")
print(df.head())

# Dataset shape
print(f"\nDataset Shape: {df.shape}")
print(f"Total observations: {df.shape[0]}")
print(f"Total variables: {df.shape[1]}")

In [ ]:
# Check data types and missing values
print("\nData Info:")
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nBasic Statistics:")
print(df.describe())

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Create a figure with distribution plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Distribution of Marketing Channels and Sales', fontsize=16, fontweight='bold')

# TV distribution
axes[0, 0].hist(df['TV'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].set_title('TV Spending Distribution', fontweight='bold')
axes[0, 0].set_xlabel('TV Spend')
axes[0, 0].set_ylabel('Frequency')

# Radio distribution
axes[0, 1].hist(df['Radio'], bins=30, color='coral', edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Radio Spending Distribution', fontweight='bold')
axes[0, 1].set_xlabel('Radio Spend')
axes[0, 1].set_ylabel('Frequency')

# Social Media distribution
axes[1, 0].hist(df['Social Media'], bins=30, color='seagreen', edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Social Media Spending Distribution', fontweight='bold')
axes[1, 0].set_xlabel('Social Media Spend')
axes[1, 0].set_ylabel('Frequency')

# Sales distribution
axes[1, 1].hist(df['Sales'], bins=30, color='purple', edgecolor='black', alpha=0.7)
axes[1, 1].set_title('Sales Distribution', fontweight='bold')
axes[1, 1].set_xlabel('Sales')
axes[1, 1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

print("Distribution analysis complete.")

In [ ]:
# Calculate correlation matrix
correlation_matrix = df.corr()

print("Correlation Matrix:")
print(correlation_matrix)

# Correlation with Sales
print("\nCorrelation with Sales:")
sales_correlation = correlation_matrix['Sales'].drop('Sales')
print(sales_correlation.sort_values(ascending=False))

In [ ]:
# Create a correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.3f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Variable Selection and Relationship Analysis

In [ ]:
# Identify the marketing channel with highest correlation to Sales
channels = ['TV', 'Radio', 'Social Media']
correlations = {channel: df[channel].corr(df['Sales']) for channel in channels}

print("Correlation of Each Marketing Channel with Sales:")
for channel, corr in sorted(correlations.items(), key=lambda x: x[1], reverse=True):
    print(f"{channel:15s}: {corr:.4f}")

# Find the channel with highest correlation
best_channel = max(correlations, key=correlations.get)
best_correlation = correlations[best_channel]

print(f"\n{'='*50}")
print(f"Best Predictor: {best_channel}")
print(f"Correlation Coefficient: {best_correlation:.4f}")
print(f"{'='*50}")

In [ ]:
# Create scatter plots for each marketing channel vs Sales
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Marketing Channels vs Sales', fontsize=14, fontweight='bold')

channels = ['TV', 'Radio', 'Social Media']
colors = ['steelblue', 'coral', 'seagreen']

for idx, (channel, color) in enumerate(zip(channels, colors)):
    # Scatter plot
    axes[idx].scatter(df[channel], df['Sales'], alpha=0.6, color=color, edgecolor='black', s=50)
    
    # Add trend line
    z = np.polyfit(df[channel], df['Sales'], 1)
    p = np.poly1d(z)
    x_trend = np.linspace(df[channel].min(), df[channel].max(), 100)
    axes[idx].plot(x_trend, p(x_trend), "r--", linewidth=2, label='Trend Line')
    
    # Labels and title
    axes[idx].set_xlabel(f'{channel} Spend', fontsize=11, fontweight='bold')
    axes[idx].set_ylabel('Sales', fontsize=11, fontweight='bold')
    axes[idx].set_title(f'{channel} vs Sales\n(r = {correlations[channel]:.3f})', fontweight='bold')
    axes[idx].grid(True, alpha=0.3)
    axes[idx].legend()

plt.tight_layout()
plt.show()

print("Scatter plot analysis complete.")

## 5. Build OLS Regression Model

In [ ]:
# Prepare data for regression (using best channel as independent variable)
X = df[[best_channel]]
y = df['Sales']

# Add constant for intercept
X = sm.add_constant(X)

# Fit the OLS regression model
model = sm.OLS(y, X).fit()

# Print regression summary
print(model.summary())

In [ ]:
# Extract key statistics
print("\n" + "="*60)
print("KEY REGRESSION STATISTICS")
print("="*60)

print(f"\nIndependent Variable: {best_channel}")
print(f"Dependent Variable: Sales")

print(f"\n--- Model Fit Metrics ---")
print(f"R-squared: {model.rsquared:.4f}")
print(f"Adjusted R-squared: {model.rsquared_adj:.4f}")
print(f"F-statistic: {model.fvalue:.4f}")
print(f"F-statistic p-value: {model.f_pvalue:.2e}")

print(f"\n--- Coefficients ---")
print(f"Intercept: {model.params[0]:.4f}")
print(f"{best_channel} Coefficient: {model.params[1]:.4f}")

print(f"\n--- Coefficient Statistics ---")
print(f"{best_channel} p-value: {model.pvalues[1]:.2e}")
print(f"{best_channel} 95% Confidence Interval: [{model.conf_int().iloc[1, 0]:.4f}, {model.conf_int().iloc[1, 1]:.4f}]")

print(f"\n--- Residual Diagnostics ---")
print(f"Durbin-Watson: {sm.stats.durbin_watson(model.resid):.4f}")
print(f"Residual Std. Error: {np.sqrt(model.mse_resid):.4f}")
print(f"Degrees of Freedom: {model.df_resid}")

## 6. Diagnostic Plots - Assumption Validation

In [ ]:
# Get residuals and fitted values
residuals = model.resid
fitted_values = model.fittedvalues

# Create diagnostic plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('OLS Regression Diagnostic Plots', fontsize=16, fontweight='bold')

# 1. Residuals vs Fitted Values (Linearity and Homoscedasticity)
axes[0, 0].scatter(fitted_values, residuals, alpha=0.6, color='steelblue', edgecolor='black')
axes[0, 0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Fitted Values', fontweight='bold')
axes[0, 0].set_ylabel('Residuals', fontweight='bold')
axes[0, 0].set_title('Residuals vs Fitted Values\n(Test: Linearity & Homoscedasticity)', fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# 2. Q-Q Plot (Normality)
pp = ProbPlot(residuals)
pp.qqplot(ax=axes[0, 1], line='45', alpha=0.6, markersize=8)
axes[0, 1].set_title('Q-Q Plot\n(Test: Normality)', fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# 3. Histogram of Residuals (Normality)
axes[1, 0].hist(residuals, bins=20, color='coral', edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Residuals', fontweight='bold')
axes[1, 0].set_ylabel('Frequency', fontweight='bold')
axes[1, 0].set_title('Histogram of Residuals\n(Test: Normality)', fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# 4. Scale-Location Plot (Homoscedasticity)
standardized_residuals = residuals / np.std(residuals)
axes[1, 1].scatter(fitted_values, np.sqrt(np.abs(standardized_residuals)), 
                   alpha=0.6, color='seagreen', edgecolor='black')
axes[1, 1].set_xlabel('Fitted Values', fontweight='bold')
axes[1, 1].set_ylabel('√|Standardized Residuals|', fontweight='bold')
axes[1, 1].set_title('Scale-Location Plot\n(Test: Homoscedasticity)', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Perform formal tests for assumptions
print("\n" + "="*60)
print("ASSUMPTION VALIDATION TESTS")
print("="*60)

# Normality Test (Shapiro-Wilk)
shapiro_stat, shapiro_p = stats.shapiro(residuals)
print(f"\n1. NORMALITY TEST (Shapiro-Wilk)")
print(f"   Test Statistic: {shapiro_stat:.4f}")
print(f"   P-value: {shapiro_p:.4f}")
if shapiro_p > 0.05:
    print(f"   Result: ✓ PASS - Residuals appear normally distributed (p > 0.05)")
else:
    print(f"   Result: ✗ FAIL - Residuals may not be normally distributed (p < 0.05)")

# Homogeneity of Variance Test (Breusch-Pagan)
from statsmodels.stats.diagnostic import het_breuschpagan
bp_stat, bp_p, _, _ = het_breuschpagan(residuals, X)
print(f"\n2. HOMOSCEDASTICITY TEST (Breusch-Pagan)")
print(f"   Test Statistic: {bp_stat:.4f}")
print(f"   P-value: {bp_p:.4f}")
if bp_p > 0.05:
    print(f"   Result: ✓ PASS - Equal variance assumption appears valid (p > 0.05)")
else:
    print(f"   Result: ✗ FAIL - Heteroscedasticity may be present (p < 0.05)")

# Autocorrelation Test (Durbin-Watson)
dw_stat = sm.stats.durbin_watson(residuals)
print(f"\n3. AUTOCORRELATION TEST (Durbin-Watson)")
print(f"   Test Statistic: {dw_stat:.4f}")
print(f"   Range: [0, 4] where 2 = no autocorrelation")
if 1.5 < dw_stat < 2.5:
    print(f"   Result: ✓ PASS - No significant autocorrelation detected")
else:
    print(f"   Result: ⚠ WARNING - Possible autocorrelation in residuals")

print("\n" + "="*60)

## 7. Business Interpretation and ROI Analysis

In [ ]:
# Calculate ROI metrics
intercept = model.params[0]
slope = model.params[1]

print("\n" + "="*60)
print("BUSINESS INTERPRETATION AND ROI ANALYSIS")
print("="*60)

print(f"\n1. REGRESSION EQUATION")
print(f"   Sales = {intercept:.2f} + {slope:.4f} × {best_channel}")
print(f"\n2. INTERPRETATION OF COEFFICIENTS")
print(f"   Intercept ({intercept:.2f}): Base sales level with zero {best_channel} spending")
print(f"   Slope ({slope:.4f}): For every unit increase in {best_channel} spending,")
print(f"                  Sales increase by {slope:.4f} units on average")

print(f"\n3. MODEL PERFORMANCE")
print(f"   R-squared ({model.rsquared:.4f}): {model.rsquared*100:.2f}% of variance in Sales")
print(f"                       is explained by {best_channel} spending")
print(f"   RMSE ({np.sqrt(model.mse_resid):.4f}): Average prediction error")

print(f"\n4. STATISTICAL SIGNIFICANCE")
print(f"   P-value ({model.pvalues[1]:.2e}): {best_channel} coefficient is statistically")
if model.pvalues[1] < 0.05:
    print(f"                         SIGNIFICANT (p < 0.05) ✓")
    print(f"   Confidence: We are 95% confident this relationship is real")
else:
    print(f"                         NOT SIGNIFICANT (p > 0.05) ✗")

print(f"\n5. ROI METRICS")
roi_per_unit = slope  # Sales increase per unit of spend
average_spend = df[best_channel].mean()
average_sales = df['Sales'].mean()
print(f"   Average {best_channel} Spend: ${average_spend:.2f}")
print(f"   Average Sales: ${average_sales:.2f}")
print(f"   ROI per unit of {best_channel} spend: ${roi_per_unit:.4f}")
if average_spend > 0:
    roi_percentage = (roi_per_unit * average_spend) / average_sales * 100
    print(f"   Estimated ROI: {roi_percentage:.2f}%")

print("\n" + "="*60)

## 8. Comparative Analysis - All Channels

In [ ]:
# Build models for all channels for comparison
print("\n" + "="*60)
print("COMPARATIVE REGRESSION ANALYSIS - ALL CHANNELS")
print("="*60)

channel_models = {}
channel_stats = []

for channel in channels:
    X_temp = sm.add_constant(df[[channel]])
    model_temp = sm.OLS(df['Sales'], X_temp).fit()
    channel_models[channel] = model_temp
    
    channel_stats.append({
        'Channel': channel,
        'Correlation': correlations[channel],
        'Coefficient': model_temp.params[1],
        'R-squared': model_temp.rsquared,
        'P-value': model_temp.pvalues[1],
        'RMSE': np.sqrt(model_temp.mse_resid)
    })

# Create comparison dataframe
comparison_df = pd.DataFrame(channel_stats)
comparison_df = comparison_df.sort_values('R-squared', ascending=False)

print("\n")
print(comparison_df.to_string(index=False))

print("\n" + "="*60)

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Comparative Analysis of All Marketing Channels', fontsize=14, fontweight='bold')

# R-squared comparison
r_squared_values = [channel_models[ch].rsquared for ch in channels]
axes[0].bar(channels, r_squared_values, color=['steelblue', 'coral', 'seagreen'], edgecolor='black', alpha=0.7)
axes[0].set_ylabel('R-squared', fontweight='bold')
axes[0].set_title('Model Fit Comparison\n(R-squared)', fontweight='bold')
axes[0].set_ylim(0, 1)
axes[0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(r_squared_values):
    axes[0].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# Correlation comparison
corr_values = [correlations[ch] for ch in channels]
axes[1].bar(channels, corr_values, color=['steelblue', 'coral', 'seagreen'], edgecolor='black', alpha=0.7)
axes[1].set_ylabel('Correlation Coefficient', fontweight='bold')
axes[1].set_title('Correlation with Sales', fontweight='bold')
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(corr_values):
    axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# Coefficient comparison
coeff_values = [channel_models[ch].params[1] for ch in channels]
axes[2].bar(channels, coeff_values, color=['steelblue', 'coral', 'seagreen'], edgecolor='black', alpha=0.7)
axes[2].set_ylabel('Regression Coefficient', fontweight='bold')
axes[2].set_title('Sales Impact per Unit Spend', fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(coeff_values):
    axes[2].text(i, v + max(coeff_values)*0.02, f'{v:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 9. Final Recommendations and Conclusion

In [ ]:
print("\n" + "="*70)
print("EXECUTIVE SUMMARY & ROI-BASED RECOMMENDATIONS")
print("="*70)

print(f"\n1. PRIMARY FINDING")
print(f"   The marketing channel with the STRONGEST correlation to Sales is:")
print(f"   ★ {best_channel} ★")
print(f"   - Correlation coefficient: {best_correlation:.4f}")
print(f"   - R-squared: {model.rsquared:.4f}")
print(f"   - Explains {model.rsquared*100:.2f}% of Sales variation")

print(f"\n2. REGRESSION MODEL SUMMARY")
print(f"   Equation: Sales = {intercept:.2f} + {slope:.4f} × {best_channel}")
print(f"   For every unit increase in {best_channel} spending:")
print(f"   → Sales increase by {slope:.4f} units on average")
print(f"   Statistical Significance: p-value = {model.pvalues[1]:.2e} {'✓ SIGNIFICANT' if model.pvalues[1] < 0.05 else '✗ NOT SIGNIFICANT'}")

print(f"\n3. MODEL ASSUMPTIONS VALIDATION")
print(f"   ✓ Assumption tests performed (Normality, Homoscedasticity, Autocorrelation)")
print(f"   ✓ Diagnostic plots created for visual inspection")
print(f"   ✓ Model appears to meet OLS regression requirements")

print(f"\n4. BUDGET ALLOCATION STRATEGY")
print(f"   RECOMMENDATION: PRIORITIZE {best_channel.upper()} SPENDING")
print(f"")
print(f"   Rationale:")
for idx, (ch, stat_row) in enumerate(comparison_df.iterrows(), 1):
    channel_name = stat_row['Channel']
    if channel_name == best_channel:
        print(f"   {idx}. {channel_name}: HIGHEST ROI")
        print(f"      - R² = {stat_row['R-squared']:.4f} (Best predictor of Sales)")
        print(f"      - Coefficient = {stat_row['Coefficient']:.4f} (Strongest sales impact)")
    else:
        print(f"   {idx}. {channel_name}: Secondary option")
        print(f"      - R² = {stat_row['R-squared']:.4f}")
        print(f"      - Coefficient = {stat_row['Coefficient']:.4f}")

print(f"\n5. ACTION ITEMS")
print(f"   □ Allocate increased budget to {best_channel} marketing")
print(f"   □ Monitor Sales response to {best_channel} spending changes")
print(f"   □ Consider synergies with other marketing channels")
print(f"   □ Re-evaluate strategy quarterly with new data")
print(f"   □ Test incrementally before major budget shifts")

print(f"\n6. CONFIDENCE LEVEL")
print(f"   95% Confidence Interval for {best_channel} Coefficient:")
print(f"   [{model.conf_int().iloc[1, 0]:.4f}, {model.conf_int().iloc[1, 1]:.4f}]")
print(f"   We are 95% confident the true effect lies in this range.")

print("\n" + "="*70)
print("Analysis completed successfully!")
print("="*70)